# 1. Difference between map() and flatMap()

**map()** gives one output for each input element.

**flatMap()** can give multiple outputs for one input and then flattens the result.

map() gives one output per input, while flatMap() can give multiple outputs per input.

In [ ]:
# map()

from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('SparkByExamples.com').getOrCreate()
sc = spark.sparkContext
rdd = sc.parallelize([1, 2, 3])
print(rdd.map(lambda x: x * 2).collect())

# flatMap()
rdd = sc.parallelize(["hello world", "spark python"])
print(rdd.flatMap(lambda x: x.split(" ")).collect())

[2, 4, 6]
['hello', 'world', 'spark', 'python']


## 2. How does collect() work in RDD?

collect() brings all data from RDD partitions to the driver program.

collect() should be used carefully because if the RDD contains huge amount of data, i can cause memory problem

In [ ]:
rdd = sc.parallelize([1, 2, 3, 4])
print(rdd.collect())

[1, 2, 3, 4]


## 3. Difference between reduceByKey() and groupByKey()

**reduceByKey()** combines values with the same key using a function.

**groupByKey()** groups all values belonging to the same key.


In [ ]:
data = [("A", 10), ("A", 20), ("B", 30)]
rdd = sc.parallelize(data)
print("reduceByKey:", rdd.reduceByKey(lambda a, b: a + b).collect())
print("groupByKey:", [(k, list(v)) for k, v in rdd.groupByKey().collect()])

reduceByKey: [('A', 30), ('B', 30)]
groupByKey: [('A', [10, 20]), ('B', [30])]


## 4. What are RDD inbuilt fuctions?

Actions trigger execution and return a result or save data.

Common actions: collect(), count(), first(), take(), reduce(), saveAsTextFile(), foreach().

Transformations are lazy. Actions trigger execution.

## 5. Explain about inferschema in csv file?

When Spark reads a CSV file, it can automatically determine the data types of columns.

Disadvantages:
1.It can increase processing time.
2.Sometimes Spark may infer the wrong data type.

In [ ]:
df = spark.read.option("header", "true").option("inferSchema", "true").csv("data.csv")
df.printSchema()

## 6. How does withColumn() work?

withColumn() is used to create a new column or modify an existing column in a DataFrame.

In [ ]:
from pyspark.sql.functions import col

# Create a new column
df = df.withColumn("salary_bonus", col("salary") * 0.10)

# Modify an existing column
df = df.withColumn("salary", col("salary") + 1000)

## 7. Explain Spark write modes

- **append**: adds new data.
- **overwrite**: replaces existing data.
- **error**: throws an error if the destination exists.
- **ignore**: does nothing if the file is exists.

In [ ]:
df.write.mode("append").parquet("output/")
df.write.mode("overwrite").parquet("output/")
df.write.mode("error").parquet("output/")
df.write.mode("ignore").parquet("output/")

## 8. Difference between withColumn() and withColumnRenamed()

withColumn() creates or modifies a column.

withColumnRenamed() only renames a column.



In [ ]:
df = df.withColumn("salary", col("salary") * 2)
df = df.withColumnRenamed("name", "customer_name")

## 9. Difference between CSV, Avro, Parquet, and ORC



 **CSV** :- Text-based and human-readable

 **Avro**:- Row-based format with schema support.  

 **Parquet** :- Column-based and very suitable for analytics.

 **ORC** :- Column-based good for big Data Processing

 Parquet is very commonly used Data Engineering for data lakes and Spark processing.

## 10. Difference between Partitioning and Bucketing

**Partitioning** divides data into  based on column values, such as 'country=India'

**Bucketing** distributes data into a fixed number of buckets based on a hash of a column.



In [ ]:
# Partitioned write
df.write.mode("overwrite").partitionBy("country").parquet("sales_partitioned/")

# Bucketed table
df.write.mode("overwrite").bucketBy(10, "customer_id").saveAsTable("customer_bucketed")

## 11. Difference between Spark SQL and DataFrame API

**Spark SQL** Allows you to query data using SQL

**DataFrame API** uses PySpark functions and methods.

Both use for Spark's underlying execution engine.

In [ ]:
# Spark SQL
df.createOrReplaceTempView("employees")
spark.sql("""SELECT name, salary FROM employees WHERE salary > 50000""").show()

# DataFrame API
df.select("name", "salary").filter(col("salary") > 50000).show()

## 12. How do you create a table from a DataFrame?

Create a temporary view using createOrReplaceTempView() and query it with Spark SQL.

For a permanent Spark table, use saveAsTable().

In [ ]:
# Temporary view
df.createOrReplaceTempView("employees")
spark.sql("SELECT * FROM employees").show()

# Permanent table
df.write.mode("overwrite").saveAsTable("employees")
spark.sql("SELECT * FROM employees").show()